In [ ]:
### classification 의 linear layer 를 LoRA layer 로 교체하는 파인 튜닝
  - LoRALayer class
  - LinearWithLoRA class
  - replace_linear_with_lora()
  - 파인 튜닝 모델 로딩 / 튜닝 / 학습 / 평가

In [ ]:
### LoRA 를 적용하는 Layer

class LoRALayer(nn.Module) :
    # LoRA rank Adaptation Layer

    def __init__( self, d_in, d_out, rank, alpha ) :
        super().__init__()

        ## 1. A 행렬 초기화
        self.A = nn.Parameter( torch.empty( d_in, rank ) )      # A : [d_in, rank]
        nn.init.kaiming_uniform( self.A, a=math.sqrt(5) )       # 정규 분포를 가지는 값으로 랜덤하게 설정

        ## 2. B 행렬 초기화
        self.B = nn.Parameter( torch.zeros( rank, d_out ) )     # B : [rank, d_out], 0 으로 초기화

        ## 3. rank, alpha 저장
        self.rank = rank
        self.alpha = alpha

    def forward(self, x):
        # LoRA Adaptor 적용
        x = (self.alpha / self.rank) * ( x @ self.A @ self.B )      # (a / R) * ( x @ A @ B )
        return x

In [ ]:
### LinearWithLoRA Layer
  - 기존 Linear 레이어를 감싸서 LoRA 어댑터를 추가한 클래스
  - 기존 Linear 출력에 LoRA 를 더해서 리턴

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()

        ## linear 와 lora layer 생성
        self.linear = linear
        self.lora = LoRALayer( linear._in_features, linear.out_features, rank, alpha )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [ ]:
### replace_linear_with_lora()

def replace_linear_with_lora( model, rank, alpha ) :
    for name, module in model.named_children() :
        if isinstance( module, torch.nn.Linear ) :
            setattr( model, name, LinearWithLoRA( module, rank, alpah ) )
        else:
            # 재귀 호출
            replace_linear_with_lora( module, rank, alpha )

In [ ]:
### 모델 로드 / LoRA 튜닝 / 학습 / 평가  (ham/spam)

torch.manual_seed(123)
device = torch.device( "cuda" if torch.cuda.is_available() else "cpu" )

## 1. 데이터 셋 준비
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
balanced_df = create_balanced_dataset( df )
balanced_df["Label"] = balanced_df["Lable"].map({ "ham": 0, "spam":1 })

train_df, val_df, test_df = random_split( balanced_df, 0.7, 0.1 )
train_df.to_csv("datas/train.csv", index=None)
val_df.to_csv("datas/val.csv", index=None)
test_df.to_csv("datas/test.csv", index=None)

tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("datas/train.csv", max_length=None, tokenizer=tokenizer)                        # max_length : None
val_dataset = SpamDataset("datas/val.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)        # max_length : train_dataset.max_length
test_dataset = SpamDataset("datas/test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)      # max_length : train_dataset.max_length

train_loader = DataLoader( train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True )             # shuffle/drop_last : True
val_loader = DataLoader( val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False )               # shuffle/drop_last : False
test_loader = DataLoader( test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False )             # shuffle/drop_last : False


## 2. 모델 로딩 및 파인 튜닝 (LoRA)
# 2.1 모델 로딩
model = GPTModel(BASE_CONFIG)
checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint)
model.to(device)

# 2.2 출력 헤드 교체 (2개 분류)
num_classes = 2
model.out_head = torch.nn.Linear( in_features=768, out_features=num_classes )

# 2.3 모델의 모든 파라미터 학습 미적용
for param in model.parameters() :
    param.requires_grad = False

# 2.4 모든 Linear 레이어에 LoRA 적용 (파인 튜닝)
replace_linear_with_lora( model, rank=LORA_RANK, alpah=LORA_ALPHA )

model.to(device)


## 3. 학습 가능 파라미터 확인 및 학습
total_params = sum( p.numel() for p in model.parameters() if p.requires_grad )
print( f"총 학습 가능 파라미터 수 : {total_params}" )

optimizer = torch.optim.AdamW( model.parameters(), lr=LEARNING_RATE, weight_decay=0.1 )

stat_time = time.time()
train_losses, val_losses, train_accs, val_accs = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device, num_epochs=NUM_EPOCHS, eval_freq=50, eval_iter=5
)
end_time = time.time()
excution_time = (end_time - start_time) / 60


## 4. 최종 평가
train_accs = calc_accuracy_loader( train_loader, model, device )
val_accs = calc_accuracy_loader( val_loader, model, device )
test_accs = calc_accuracy_loader( test_loader, model, device )

print( f"훈련 정확도: {train_accs * 100 : .2f}" )
print( f"검증 정확도: {val_accs * 100 : .2f}" )
print( f"테스트 정확도: {test_accs * 100 : .2f}" )